# Flow Assurance Starter

**Task:** [Replace with task title]
**Date:** [YYYY-MM-DD]

Template for hydrate prediction, pipeline hydraulics, and wax/corrosion screening.

In [ ]:
# ── NeqSim Setup (dual-boot: devtools or pip) ──
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

import matplotlib.pyplot as plt
import numpy as np
import json, os, pathlib

NOTEBOOK_DIR = pathlib.Path(globals().get(
    "__vsc_ipynb_file__", os.path.abspath("step2_analysis/01_flow_assurance.ipynb")
)).resolve().parent
TASK_DIR = NOTEBOOK_DIR.parent
FIGURES_DIR = TASK_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
# ── Import NeqSim classes ──
if NEQSIM_MODE == "pip":
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
    ProcessSystem = jneqsim.process.processmodel.ProcessSystem
    Stream = jneqsim.process.equipment.stream.Stream
    PipeBeggsAndBrills = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills
    print("Classes imported from jneqsim")

## 1. Fluid Definition

**Important:** Use SRK-CPA for systems with water — standard SRK underestimates water
solubility by ~1000x.

In [ ]:
# ── Create fluid with water (use CPA) ──
fluid = SystemSrkCPAstatoil(273.15 + 25.0, 80.0)

fluid.addComponent("methane", 0.80)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.05)
fluid.addComponent("n-butane", 0.03)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("water", 0.02)

fluid.setMixingRule(10)  # CPA mixing rule
fluid.setMultiPhaseCheck(True)

print("Fluid: {} components (CPA EOS)".format(fluid.getNumberOfComponents()))

## 2. Hydrate Formation Temperature

In [ ]:
# ── Hydrate equilibrium curve ──
pressures = [20, 40, 60, 80, 100, 120, 150, 200]
hydrate_temps = []

for P in pressures:
    test_fluid = fluid.clone()
    test_fluid.setPressure(float(P))
    test_fluid.setTemperature(273.15 + 5.0)  # Initial guess
    ops = ThermodynamicOperations(test_fluid)
    try:
        ops.hydrateFormationTemperature()
        T_hyd = test_fluid.getTemperature() - 273.15
        hydrate_temps.append(T_hyd)
        print("P = {:>6.0f} bara  ->  T_hyd = {:>6.1f} degC".format(P, T_hyd))
    except Exception as e:
        hydrate_temps.append(None)
        print("P = {:>6.0f} bara  ->  FAILED: {}".format(P, e))

In [ ]:
# ── Plot hydrate curve ──
valid = [(p, t) for p, t in zip(pressures, hydrate_temps) if t is not None]
if valid:
    p_valid, t_valid = zip(*valid)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(t_valid, p_valid, 'b-o', linewidth=2, markersize=6, label='Hydrate curve')
    ax.fill_betweenx(p_valid, -20, t_valid, alpha=0.15, color='blue', label='Hydrate region')
    ax.set_xlabel('Temperature (\u00b0C)')
    ax.set_ylabel('Pressure (bara)')
    ax.set_title('Hydrate Formation Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / "hydrate_curve.png"), dpi=150, bbox_inches="tight")
    plt.show()

### Discussion

**Observation:** [Hydrate curve description]

**Physical mechanism:** [Why hydrates form at these conditions]

**Engineering implication:** [Subcooling margin, inhibitor requirements]

**Recommendation:** [Actions]

## 3. Pipeline Pressure/Temperature Profile

In [ ]:
# ── Pipeline simulation ──
process = ProcessSystem()

feed = Stream("Pipeline Inlet", fluid)
feed.setFlowRate(50000.0, "kg/hr")
feed.setTemperature(60.0, "C")
feed.setPressure(100.0, "bara")
process.add(feed)

pipe = PipeBeggsAndBrills("Export Pipeline", feed)
pipe.setPipeWallRoughness(5e-5)  # m
pipe.setLength(50000.0)  # 50 km
pipe.setDiameter(0.3048)  # 12 inch
pipe.setAngle(0.0)  # Horizontal
pipe.setNumberOfIncrements(50)
# pipe.setOuterTemperature(273.15 + 4.0)  # Seabed temperature
process.add(pipe)

process.run()

outlet_T = pipe.getOutletStream().getTemperature() - 273.15
outlet_P = pipe.getOutletStream().getPressure()
print("Pipeline outlet: {:.1f} degC, {:.1f} bara".format(outlet_T, outlet_P))

## 4. Save Results

In [ ]:
results_path = TASK_DIR / "results.json"
if results_path.exists():
    with open(results_path, "r") as f:
        results = json.load(f)
else:
    results = {}

results["key_results"] = {
    "hydrate_T_at_100bara_C": hydrate_temps[pressures.index(100)] if 100 in pressures else None,
    "pipeline_outlet_T_C": outlet_T,
    "pipeline_outlet_P_bara": outlet_P,
    "pipeline_dP_bar": 100.0 - outlet_P,
}
results["approach"] = "SRK-CPA EOS, Beggs and Brill pipeline correlation."
results["figure_captions"] = {
    "hydrate_curve.png": "Hydrate formation temperature vs pressure.",
}

with open(str(results_path), "w") as f:
    json.dump(results, f, indent=2)

print("results.json saved")